# Credit Score — AED, Cleanup, Pipeline e Modelagem

Este é o notebook principal do projeto **Credit Score**, preparado para rodar diretamente no Google Colab com o arquivo `train.csv` no mesmo diretório.

## Protocolo Experimental

- **Seed global:** `RANDOM_STATE = 42`.
- **Split:** 80% treino / 20% teste com `stratify=TARGET`.
- **Validação cruzada:** `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.
- **Métrica de seleção:** `average_precision` (PR-AUC), mais informativa para inadimplentes em base desbalanceada.
- **Métricas reportadas:** accuracy, precision, recall, F1, ROC-AUC, average precision e matriz de confusão.
- **Baseline:** `DummyClassifier`, para mostrar por que acurácia isolada é insuficiente.
- **Modelos exigidos:** KNN e Árvore de Decisão comparados via `GridSearchCV`.

## Decisões Fechadas

1. Usar `train.csv` bruto, não `cleaned_train.csv`, para evitar leakage e double-scaling.
2. Substituir `-99`, `-9998` e `-9999` por `NaN`; isso é determinístico e seguro antes do split.
3. Remover `HS_CPF`, `ORIENTACAO_SEXUAL` e `RELIGIAO` da modelagem.
4. Manter colunas `CASA` com uma flag consolidada se o padrão de ausência for compartilhado.
5. Manter `ANOSULTIMADECLARACAO` inicialmente, com flag de ausência, e validar essa decisão.
6. Usar `ColumnTransformer` dentro de `Pipeline`, com imputação e normalização fitadas somente no treino.

## 1. Imports e Constantes

Este bloco importa as bibliotecas usadas no projeto e define constantes globais. Centralizar essas definições facilita reprodutibilidade, especialmente no Colab.

`FAST_MODE` permite reduzir alguns grids durante desenvolvimento. Para uma execução final do projeto, mantenha `FAST_MODE = False`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn"])
    import seaborn as sns

from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    PrecisionRecallDisplay,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", palette="Set2")

RANDOM_STATE = 42
SENTINELS = [-99, -9998, -9999]
TARGET_COL = "TARGET"
FAST_MODE = False

## 2. Carregamento do Dataset

Este bloco carrega o `train.csv` sem aplicar transformação. A AED inicial deve refletir o estado real do arquivo recebido, incluindo os códigos sentinela.

A função procura o CSV no diretório atual e também em subpastas, o que facilita a execução tanto local quanto no Colab.

In [ ]:
def find_train_csv() -> Path:
    candidates = [Path("train.csv"), Path("/content/train.csv")]
    candidates.extend(Path.cwd().glob("**/train.csv"))
    for candidate in candidates:
        if candidate.exists() and candidate.is_file():
            return candidate
    raise FileNotFoundError("Não encontrei train.csv. Deixe o arquivo no mesmo diretório do notebook.")

DATA_PATH = find_train_csv()
df_raw = pd.read_csv(DATA_PATH)

print(f"Arquivo carregado: {DATA_PATH.resolve()}")
print(f"Shape bruto: {df_raw.shape[0]:,} linhas x {df_raw.shape[1]:,} colunas")
display(df_raw.head())

## 3. Apresentação Inicial da Base

Este bloco atende à rubrica de apresentação do conjunto de dados: número de instâncias, número de atributos, distribuição do `TARGET`, tipos de dados e percentual de dados faltantes.

A base não possui `NaN` explícito, mas possui valores sentinela que representam ausência ou invalidade. Por isso, o percentual real de faltantes é calculado com base em `-99`, `-9998` e `-9999`.

In [ ]:
shape_info = pd.DataFrame({
    "medida": ["instâncias", "atributos totais", "features candidatas", "target"],
    "valor": [len(df_raw), df_raw.shape[1], df_raw.shape[1] - 1, TARGET_COL],
})

target_counts = df_raw[TARGET_COL].value_counts(dropna=False).sort_index()
target_distribution = pd.DataFrame({
    "classe": target_counts.index.astype(str),
    "quantidade": target_counts.values,
    "percentual": (target_counts.values / len(df_raw) * 100).round(2),
})

sentinel_counts = df_raw.isin(SENTINELS).sum()
missing_report = pd.DataFrame({
    "coluna": df_raw.columns,
    "nan_explicito_pct": (df_raw.isna().mean() * 100).round(2).values,
    "sentinelas_pct": (sentinel_counts / len(df_raw) * 100).round(2).values,
    "sentinelas_qtd": sentinel_counts.values,
    "dtype": df_raw.dtypes.astype(str).values,
}).sort_values(["sentinelas_pct", "sentinelas_qtd"], ascending=False).reset_index(drop=True)

display(Markdown("### Dimensões")); display(shape_info)
display(Markdown("### Distribuição do TARGET")); display(target_distribution)
display(Markdown("### Tipos de dados")); display(df_raw.dtypes.value_counts().rename_axis("dtype").reset_index(name="quantidade"))
display(Markdown("### Colunas com sentinelas")); display(missing_report.query("sentinelas_qtd > 0").head(30))

print(f"NaN explícitos no arquivo bruto: {df_raw.isna().sum().sum():,}")
print(f"Percentual médio de sentinelas na matriz de dados: {sentinel_counts.sum() / df_raw.size * 100:.2f}%")

## 4. Descrição Textual de Atributos

A rubrica pede a descrição textual de pelo menos 10 atributos. A tabela abaixo descreve variáveis centrais para o problema de crédito, incluindo atributos individuais, domiciliares, territoriais e variáveis removidas por restrição ética/legal.

In [ ]:
attribute_descriptions = pd.DataFrame([
    ("TARGET", "Alvo: 0 indica quitação integral; 1 indica inadimplência."),
    ("HS_CPF", "Identificador do cliente; removido da modelagem."),
    ("TEMPOCPF", "Tempo associado ao CPF/cadastro, possível sinal de estabilidade cadastral."),
    ("DISTCENTROCIDADE", "Distância até o centro da cidade."),
    ("DISTZONARISCO", "Distância até zona de risco."),
    ("QTDENDERECO", "Quantidade de endereços registrados."),
    ("QTDCELULAR", "Quantidade de celulares vinculados ao cliente."),
    ("QTDFONEFIXO", "Quantidade de telefones fixos vinculados."),
    ("ESTIMATIVARENDA", "Estimativa de renda individual."),
    ("MEDIARENDACEP", "Renda média estimada da região/CEP."),
    ("IDHMUNICIPIO", "Indicador de desenvolvimento humano do município."),
    ("PIBMUNICIPIO", "Indicador econômico do município."),
    ("QTDPESSOASCASA", "Quantidade estimada de pessoas na residência."),
    ("MEDIARENDACASA", "Renda média estimada da residência."),
    ("ANOSULTIMADECLARACAO", "Anos desde a última declaração fiscal identificada."),
    ("ORIENTACAO_SEXUAL", "Variável sensível; analisada na AED e removida da modelagem."),
    ("RELIGIAO", "Variável sensível; analisada na AED e removida da modelagem."),
], columns=["atributo", "descrição"])

display(attribute_descriptions)

## 5. Cópia para AED com Sentinelas como `NaN`

Para estatísticas e gráficos, os sentinelas precisam ser tratados como ausentes; caso contrário, valores como `-9999` distorcem histogramas e boxplots. Criamos `df_eda` para visualização e análise estatística, mantendo `df_raw` intacto para comprovar o estado original do arquivo.

Essa etapa ainda não é modelagem: é uma conversão determinística para permitir AED válida.

In [ ]:
df_eda = df_raw.replace(SENTINELS, np.nan).copy()
df_eda[TARGET_COL] = df_eda[TARGET_COL].astype(int)

LEGAL_SENSITIVE_COLS = ["ORIENTACAO_SEXUAL", "RELIGIAO"]
CASA_COLS = [
    "QTDPESSOASCASA", "MENORRENDACASA", "MAIORRENDACASA", "SOMARENDACASA",
    "MEDIARENDACASA", "MAIORIDADECASA", "MENORIDADECASA", "MEDIAIDADECASA",
    "INDICMENORDEIDADE", "COBRANCABAIXOCASA", "COBRANCAMEDIOCASA",
    "COBRANCAALTACASA", "SEGMENTACAOFINBAIXACASA", "SEGMENTACAOFINMEDIACASA",
    "SEGMENTACAOALTACASA", "BOLSAFAMILIACASA", "FUNCIONARIOPUBLICOCASA",
]
existing_casa_cols = [col for col in CASA_COLS if col in df_eda.columns]

df_eda["GRUPO_CASA_AUSENTE_EDA"] = df_eda[existing_casa_cols].isna().any(axis=1).astype(int)
df_eda["DECLARACAO_AUSENTE_EDA"] = df_eda["ANOSULTIMADECLARACAO"].isna().astype(int)

print(f"Cópia para AED: {df_eda.shape}")
print(f"Colunas CASA encontradas: {len(existing_casa_cols)}")

## 6. Funções Auxiliares da AED

Este bloco cria funções para padronizar as análises no formato **questão/hipótese → análise → discussão**. A padronização evita repetição e garante que as 15 análises univariadas e 5 multivariadas mantenham a mesma estrutura narrativa.

In [ ]:
def section(title: str, question: str, hypothesis: str) -> None:
    text = f"### {title}\n**Questão/Hipótese:** {question}\n\n**Hipótese inicial:** {hypothesis}"
    display(Markdown(text))


def discuss(text: str) -> None:
    display(Markdown(f"**Análise/Discussão:** {text}"))


def plot_numeric_by_target(df: pd.DataFrame, col: str, title: str, clip_quantile: float = 0.99) -> pd.DataFrame:
    valid = df[[col, TARGET_COL]].dropna().copy()
    if valid.empty:
        discuss(f"Sem valores válidos para `{col}`.")
        return pd.DataFrame()
    upper = valid[col].quantile(clip_quantile)
    valid_plot = valid[valid[col] <= upper].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.histplot(data=valid_plot, x=col, hue=TARGET_COL, stat="density", common_norm=False, bins=30, ax=axes[0])
    axes[0].set_title(f"Distribuição por TARGET — {title}")
    sns.boxplot(data=valid_plot, x=TARGET_COL, y=col, ax=axes[1])
    axes[1].set_title(f"Boxplot por TARGET — {title}")
    axes[1].set_xlabel("TARGET")
    plt.tight_layout(); plt.show()
    return valid.groupby(TARGET_COL)[col].agg(["count", "mean", "median", "std", "min", "max"])


def plot_target_rate_by_binary(df: pd.DataFrame, col: str, title: str) -> pd.DataFrame:
    summary = df.groupby(col, dropna=False)[TARGET_COL].agg(quantidade="count", taxa_inadimplencia="mean").reset_index()
    summary["taxa_inadimplencia_pct"] = (summary["taxa_inadimplencia"] * 100).round(2)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=summary, x=col, y="taxa_inadimplencia", ax=ax)
    ax.set_title(title); ax.set_ylabel("Taxa de inadimplência"); ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
    plt.show()
    return summary


def plot_category_distribution(df: pd.DataFrame, col: str, title: str, top_n: int = 12) -> pd.DataFrame:
    summary = df.groupby(col, dropna=False)[TARGET_COL].agg(quantidade="count", taxa_inadimplencia="mean").sort_values("quantidade", ascending=False).head(top_n).reset_index()
    summary["percentual_base"] = (summary["quantidade"] / len(df) * 100).round(2)
    summary["taxa_inadimplencia_pct"] = (summary["taxa_inadimplencia"] * 100).round(2)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.barplot(data=summary, y=col, x="quantidade", ax=axes[0])
    axes[0].set_title(f"Frequência — {title}")
    sns.barplot(data=summary, y=col, x="taxa_inadimplencia", ax=axes[1])
    axes[1].set_title(f"Taxa de inadimplência — {title}"); axes[1].xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    plt.tight_layout(); plt.show()
    return summary

## 7. Análises Univariadas — 15 Análises

Este bloco executa as 15 análises univariadas exigidas. Cada análise apresenta uma questão, uma hipótese, uma visualização ou estatística e uma discussão.

Algumas análises comparam a variável com `TARGET`, seguindo a prática usada em AED supervisionada: entendemos a distribuição de uma variável e verificamos se ela parece útil para discriminar a classe de inadimplência.

In [ ]:
# 1 TARGET
section("Univariada 1 — TARGET", "Qual é o desbalanceamento entre adimplentes e inadimplentes?", "A classe inadimplente é minoritária.")
fig, ax = plt.subplots(figsize=(6, 4)); sns.barplot(data=target_distribution, x="classe", y="percentual", ax=ax)
ax.set_title("Distribuição percentual do TARGET"); plt.show(); display(target_distribution)
discuss("A classe 0 domina a base. Por isso, accuracy não será usada como métrica de seleção.")

# 2 Sentinelas
section("Univariada 2 — Sentinelas", "Quais colunas têm maior percentual de ausência codificada?", "Algumas colunas têm alta taxa de -99/-9998/-9999.")
top_missing = missing_report.query("sentinelas_qtd > 0").head(20)
fig, ax = plt.subplots(figsize=(10, 7)); sns.barplot(data=top_missing, y="coluna", x="sentinelas_pct", ax=ax)
ax.set_title("Top 20 colunas por sentinelas (%)"); plt.show(); display(top_missing)
discuss("Colunas acima de 80% serão removidas; colunas com ausência relevante, mas ainda informativas, serão mantidas com flags.")

# 3-8 numéricas
for idx, col, question, hypothesis in [
    (3, "TEMPOCPF", "Clientes com maior tempo de CPF/cadastro inadimplem menos?", "Maior estabilidade cadastral pode reduzir risco."),
    (4, "ESTIMATIVARENDA", "A renda estimada difere por TARGET?", "Menor renda pode estar associada a maior inadimplência."),
    (5, "DISTZONARISCO", "A distância a zonas de risco diferencia os grupos?", "Contexto territorial pode carregar sinal."),
    (6, "DISTCENTROCIDADE", "A distância ao centro da cidade está associada ao risco?", "Distância pode refletir acesso e contexto urbano."),
    (7, "QTDCELULAR", "A quantidade de celulares diferencia os grupos?", "Contatos podem indicar estabilidade cadastral."),
    (8, "QTDENDERECO", "Quantidade de endereços sugere instabilidade cadastral?", "Muitos endereços podem indicar maior variabilidade cadastral."),
]:
    section(f"Univariada {idx} — {col}", question, hypothesis)
    summary = plot_numeric_by_target(df_eda, col, col, clip_quantile=1.0 if col in ["QTDCELULAR", "QTDENDERECO"] else 0.99)
    display(summary)
    discuss(f"`{col}` será mantida inicialmente e escalada no pipeline. Mesmo variáveis discretas afetam a distância do KNN.")

# 9-10 binárias
for idx, col, question, hypothesis in [
    (9, "FUNCIONARIOPUBLICO", "Ser funcionário público altera a taxa de inadimplência?", "Renda mais estável pode reduzir risco."),
    (10, "BOLSAFAMILIA", "Bolsa Família está associada à inadimplência?", "A variável pode capturar contexto socioeconômico; interpretação deve ser cuidadosa."),
]:
    section(f"Univariada {idx} — {col}", question, hypothesis)
    summary = plot_target_rate_by_binary(df_eda, col, f"Taxa de inadimplência por {col}")
    display(summary)
    discuss(f"`{col}` será tratada como variável numérica/binária objetiva no pipeline.")

# 11 declaração
section("Univariada 11 — ANOSULTIMADECLARACAO", "A ausência ou tempo desde declaração fiscal carrega sinal?", "Pode indicar perfil financeiro formal/informal.")
display(plot_numeric_by_target(df_eda, "ANOSULTIMADECLARACAO", "ANOSULTIMADECLARACAO", clip_quantile=1.0))
display(plot_target_rate_by_binary(df_eda, "DECLARACAO_AUSENTE_EDA", "Taxa por ausência de declaração"))
discuss("A coluna será mantida com flag e validada empiricamente, em vez de removida automaticamente.")

# 12 casa ausente
section("Univariada 12 — Ausência do grupo CASA", "A ausência de informações domiciliares muda a taxa de inadimplência?", "Ausência de informação pode ser preditiva.")
display(plot_target_rate_by_binary(df_eda, "GRUPO_CASA_AUSENTE_EDA", "Taxa por ausência do grupo CASA"))
discuss("Manteremos as colunas CASA e criaremos uma flag consolidada de ausência.")

# 13 renda casa
section("Univariada 13 — MEDIARENDACASA", "A renda média domiciliar difere por TARGET?", "Renda domiciliar menor pode aumentar risco.")
display(plot_numeric_by_target(df_eda, "MEDIARENDACASA", "MEDIARENDACASA"))
discuss("A variável tem muitos ausentes, mas valores reais em parte relevante da base.")

# 14 IDH
section("Univariada 14 — IDHMUNICIPIO", "IDH municipal diferencia os grupos?", "Contexto regional pode influenciar capacidade de pagamento.")
display(plot_numeric_by_target(df_eda, "IDHMUNICIPIO", "IDHMUNICIPIO", clip_quantile=1.0))
discuss("A variável regional será mantida por ser objetiva e agregada.")

# 15 sensíveis
section("Univariada 15 — Variáveis sensíveis", "Como ORIENTACAO_SEXUAL e RELIGIAO aparecem na base?", "Devem ser descritas na AED, mas removidas da modelagem.")
for sensitive_col in LEGAL_SENSITIVE_COLS:
    display(Markdown(f"#### {sensitive_col}"))
    display(plot_category_distribution(df_eda, sensitive_col, sensitive_col))
discuss("As variáveis sensíveis são removidas antes do treinamento por restrição ética e legal.")

## 8. Análises Multivariadas — 5 Análises

Este bloco executa as 5 análises multivariadas exigidas. Elas avaliam correlações, interações entre renda e território, decis de renda, padrões combinados de ausência e pares redundantes.

In [ ]:
# 1 Correlação selecionada
section("Multivariada 1 — Correlação", "Quais variáveis financeiras/territoriais são redundantes?", "Rendas domiciliares devem apresentar alta correlação.")
selected_corr_cols = ["TEMPOCPF", "ESTIMATIVARENDA", "MEDIARENDACEP", "MEDIARENDACASA", "MAIORRENDACASA", "SOMARENDACASA", "IDHMUNICIPIO", "PIBMUNICIPIO", "DISTCENTROCIDADE", "DISTZONARISCO", TARGET_COL]
selected_corr_cols = [col for col in selected_corr_cols if col in df_eda.columns]
plt.figure(figsize=(10, 8)); sns.heatmap(df_eda[selected_corr_cols].corr(numeric_only=True), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlação — variáveis selecionadas"); plt.show()
discuss("Correlação alta orienta discussão de redundância, mas não implica causalidade.")

# 2 Renda x IDH
section("Multivariada 2 — ESTIMATIVARENDA x IDHMUNICIPIO", "A relação renda/IDH muda por TARGET?", "Baixa renda em regiões menos desenvolvidas pode compor maior risco.")
plot_df = df_eda[["ESTIMATIVARENDA", "IDHMUNICIPIO", TARGET_COL]].dropna().copy()
plot_df = plot_df[plot_df["ESTIMATIVARENDA"] <= plot_df["ESTIMATIVARENDA"].quantile(0.99)]
sample_plot = plot_df.sample(min(len(plot_df), 8000), random_state=RANDOM_STATE)
plt.figure(figsize=(9, 5)); sns.scatterplot(data=sample_plot, x="ESTIMATIVARENDA", y="IDHMUNICIPIO", hue=TARGET_COL, alpha=0.45)
plt.title("Renda estimada x IDH por TARGET"); plt.show()
discuss("A amostra é usada só para visualização, evitando sobreposição excessiva.")

# 3 Decis de renda
section("Multivariada 3 — Decis de renda", "A inadimplência varia por faixa de renda?", "Decis inferiores podem ter maior taxa de inadimplência.")
renda_decis = df_eda[["ESTIMATIVARENDA", TARGET_COL]].dropna().copy()
renda_decis["decil_renda"] = pd.qcut(renda_decis["ESTIMATIVARENDA"], q=10, duplicates="drop")
renda_summary = renda_decis.groupby("decil_renda", observed=False)[TARGET_COL].agg(quantidade="count", taxa_inadimplencia="mean").reset_index()
plt.figure(figsize=(12, 4)); sns.lineplot(data=renda_summary, x=renda_summary.index, y="taxa_inadimplencia", marker="o")
plt.title("Taxa de inadimplência por decil de renda"); plt.xlabel("Decil"); plt.ylabel("Taxa"); plt.gca().yaxis.set_major_formatter(lambda y, _: f"{y:.0%}"); plt.show()
display(renda_summary)
discuss("A análise por decis reduz ruído individual e mostra tendência agregada.")

# 4 Ausências combinadas
section("Multivariada 4 — Ausência CASA x declaração", "Ausências simultâneas alteram inadimplência?", "Ausência simultânea pode indicar menor formalização cadastral/financeira.")
missing_combo = df_eda.groupby(["GRUPO_CASA_AUSENTE_EDA", "DECLARACAO_AUSENTE_EDA"])[TARGET_COL].agg(quantidade="count", taxa_inadimplencia="mean").reset_index()
pivot_combo = missing_combo.pivot(index="GRUPO_CASA_AUSENTE_EDA", columns="DECLARACAO_AUSENTE_EDA", values="taxa_inadimplencia")
plt.figure(figsize=(7, 4)); sns.heatmap(pivot_combo, annot=True, fmt=".2%", cmap="YlOrRd")
plt.title("Taxa por padrões de ausência"); plt.show(); display(missing_combo)
discuss("A análise justifica flags determinísticas de ausência.")

# 5 Pares correlacionados
section("Multivariada 5 — Pares de alta correlação", "Há pares com correlação absoluta >= 0,85?", "Rendas domiciliares e percentuais complementares podem ser redundantes.")
numeric_for_corr = df_eda.select_dtypes(include="number").drop(columns=[TARGET_COL], errors="ignore")
abs_corr = numeric_for_corr.corr().abs()
pairs = []
cols = abs_corr.columns.tolist()
for i, col_a in enumerate(cols):
    for col_b in cols[i + 1:]:
        value = abs_corr.loc[col_a, col_b]
        if pd.notna(value) and value >= 0.85:
            pairs.append((col_a, col_b, value))
high_corr_pairs = pd.DataFrame(pairs, columns=["feature_a", "feature_b", "abs_corr"]).sort_values("abs_corr", ascending=False)
display(high_corr_pairs.head(30))
discuss("Não removemos automaticamente, mas registramos redundâncias para discussão e possíveis versões reduzidas.")

## 9. Limpeza Determinística para Modelagem

Este bloco aplica apenas operações sem aprendizado estatístico: sentinelas para `NaN`, criação de flags e remoção de colunas proibidas ou com ausência extrema.

A validação do grupo `CASA` é feita antes de criar a flag única. Se o padrão de ausência não for totalmente compartilhado, o código usa uma flag agregada por qualquer ausência, evitando uma suposição silenciosa.

In [ ]:
df_model = df_raw.copy().replace(SENTINELS, np.nan)
df_model[TARGET_COL] = df_model[TARGET_COL].astype(int)

COLS_PROIBIDAS_LEGAIS = ["HS_CPF", "ORIENTACAO_SEXUAL", "RELIGIAO"]
COLS_ALTA_MISSINGNESS = ["ANOSULTIMARESTITUICAO", "ANOSULTIMADECLARACAOPAGAR"]
existing_casa_cols = [col for col in CASA_COLS if col in df_model.columns]

casa_missing_matrix = df_model[existing_casa_cols].isna()
all_missing_equals_any_missing = casa_missing_matrix.all(axis=1).equals(casa_missing_matrix.any(axis=1))
unique_casa_missing_patterns = casa_missing_matrix.drop_duplicates().shape[0]

print(f"Colunas CASA encontradas: {len(existing_casa_cols)}")
print(f"all(axis=1) == any(axis=1): {all_missing_equals_any_missing}")
print(f"Padrões distintos de ausência CASA: {unique_casa_missing_patterns}")

if all_missing_equals_any_missing:
    df_model["GRUPO_CASA_AUSENTE"] = df_model[existing_casa_cols[0]].isna().astype(int)
    casa_flag_decision = "flag_unica"
else:
    df_model["GRUPO_CASA_AUSENTE"] = df_model[existing_casa_cols].isna().any(axis=1).astype(int)
    casa_flag_decision = "flag_agregada_any"

df_model["DECLARACAO_AUSENTE"] = df_model["ANOSULTIMADECLARACAO"].isna().astype(int)

cols_to_drop = [col for col in COLS_PROIBIDAS_LEGAIS + COLS_ALTA_MISSINGNESS if col in df_model.columns]
df_model = df_model.drop(columns=cols_to_drop)

print(f"Decisão flag CASA: {casa_flag_decision}")
print(f"Shape após limpeza determinística: {df_model.shape}")
print("Colunas removidas:", cols_to_drop)
display(df_model.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False).rename("qtd_nan_restante").reset_index().rename(columns={"index": "coluna"}).head(30))

## 10. Validação das Discordâncias

Este bloco compara decisões que foram debatidas:

- `RobustScaler` vs `StandardScaler` para KNN;
- manter vs remover `ANOSULTIMADECLARACAO` e sua flag;
- flag consolidada de `CASA` vs muitos indicadores redundantes.

A validação usa uma amostra estratificada para controlar custo computacional, mas mantém os mesmos 5 folds do protocolo principal. Ela não substitui o GridSearchCV final; serve para justificar as escolhas de preparação.

Na primeira execução, o `StandardScaler` superou o `RobustScaler` para KNN. Por isso, o pipeline principal foi ajustado para `StandardScaler`.

In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def stratified_sample_frame(X, y, max_rows=25000, random_state=RANDOM_STATE):
    if len(X) <= max_rows:
        return X.copy(), y.copy()
    sample_idx, _ = train_test_split(X.index, train_size=max_rows, stratify=y, random_state=random_state)
    return X.loc[sample_idx].copy(), y.loc[sample_idx].copy()


def build_preprocessor_for_columns(numeric_cols, categorical_cols, scaler):
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="DESCONHECIDO")),
        ("ohe", make_ohe()),
    ])
    return ColumnTransformer([
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ], remainder="drop")

X_compare = df_model.drop(columns=[TARGET_COL])
y_compare = df_model[TARGET_COL]
X_cmp, y_cmp = stratified_sample_frame(X_compare, y_compare, max_rows=12000 if FAST_MODE else 25000)
cv_compare = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
comparison_rows = []

# RobustScaler vs StandardScaler no KNN.
for scaler_name, scaler in [("standard", StandardScaler()), ("robust", RobustScaler())]:
    numeric_cols_cmp = X_cmp.select_dtypes(include="number").columns.tolist()
    categorical_cols_cmp = X_cmp.select_dtypes(include=["object", "string"]).columns.tolist()
    pre = build_preprocessor_for_columns(numeric_cols_cmp, categorical_cols_cmp, scaler)
    model = Pipeline([
        ("preprocess", pre),
        ("clf", KNeighborsClassifier(n_neighbors=21, weights="distance", metric="manhattan")),
    ])
    scores = cross_validate(model, X_cmp, y_cmp, cv=cv_compare, scoring={"average_precision": "average_precision", "roc_auc": "roc_auc"}, n_jobs=-1)
    comparison_rows.append({
        "discordancia": "scaler_knn",
        "configuracao": scaler_name,
        "average_precision_mean": scores["test_average_precision"].mean(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
    })

# Manter vs remover ANOSULTIMADECLARACAO.
for keep_decl in [True, False]:
    X_variant = X_cmp.copy()
    if not keep_decl:
        X_variant = X_variant.drop(columns=["ANOSULTIMADECLARACAO", "DECLARACAO_AUSENTE"], errors="ignore")
    numeric_cols_variant = X_variant.select_dtypes(include="number").columns.tolist()
    categorical_cols_variant = X_variant.select_dtypes(include=["object", "string"]).columns.tolist()
    pre = build_preprocessor_for_columns(numeric_cols_variant, categorical_cols_variant, StandardScaler())
    model = Pipeline([
        ("preprocess", pre),
        ("clf", DecisionTreeClassifier(max_depth=10, class_weight="balanced", random_state=RANDOM_STATE)),
    ])
    scores = cross_validate(model, X_variant, y_cmp, cv=cv_compare, scoring={"average_precision": "average_precision", "roc_auc": "roc_auc"}, n_jobs=-1)
    comparison_rows.append({
        "discordancia": "anos_ultima_declaracao",
        "configuracao": "manter_com_flag" if keep_decl else "remover_coluna_e_flag",
        "average_precision_mean": scores["test_average_precision"].mean(),
        "roc_auc_mean": scores["test_roc_auc"].mean(),
    })

comparison_rows.append({
    "discordancia": "indicadores_casa",
    "configuracao": f"flag_manual=1; indicadores_CASA_potenciais={len(existing_casa_cols)}",
    "average_precision_mean": np.nan,
    "roc_auc_mean": np.nan,
})

comparison_report = pd.DataFrame(comparison_rows)
display(comparison_report)
display(Markdown("**Decisão operacional:** manter flag consolidada para CASA, manter `ANOSULTIMADECLARACAO` com flag salvo evidência contrária clara, e usar `StandardScaler` como padrão porque venceu a validação empírica para KNN."))

## 11. Split Treino/Teste

Este bloco cria o holdout final. A partir daqui, o conjunto de teste não será usado para escolher hiperparâmetros, imputadores, escaladores ou decisões de modelagem.

In [ ]:
X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"X_train: {X_train.shape} | TARGET=1: {y_train.mean():.4f}")
print(f"X_test:  {X_test.shape} | TARGET=1: {y_test.mean():.4f}")

## 12. ColumnTransformer

Este bloco define o pré-processamento obrigatório. As numéricas recebem `SimpleImputer(strategy='median')` e `StandardScaler`. As categóricas recebem imputação constante e `OneHotEncoder`.

A decisão inicial favorecia `RobustScaler` por causa das caudas longas. Porém, a validação empírica mostrou melhor desempenho do `StandardScaler` no KNN. Como KNN é sensível à escala e é um dos modelos obrigatórios, adotamos `StandardScaler` no pipeline principal.

Após remover `ORIENTACAO_SEXUAL` e `RELIGIAO`, é esperado que não sobrem categóricas textuais. Mesmo assim, o bloco categórico fica na estrutura para documentar o requisito da rubrica e manter robustez.

In [ ]:
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()

print(f"Features numéricas: {len(numeric_cols)}")
print(f"Features categóricas: {len(categorical_cols)}")
if not categorical_cols:
    print("Após exclusão legal das variáveis sensíveis, não restaram categóricas textuais permitidas.")

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="DESCONHECIDO")),
    ("ohe", make_ohe()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
], remainder="drop")

## 13. Baseline e Pipelines dos Modelos

Este bloco cria o baseline e os dois pipelines exigidos. O KNN não possui `class_weight`, então o grid testará pesos por distância. A árvore recebe `class_weight='balanced'` fixo no estimador, pois a base é desbalanceada.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")

# Objetos separados deixam a inspeção dos pipelines mais clara. O GridSearchCV já
# clonaria internamente o preprocessador, mas aqui evitamos qualquer confusão ao
# comparar KNN e Árvore depois do fit.
preprocessor_knn = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
], remainder="drop")

preprocessor_dt = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
], remainder="drop")

pipe_knn = Pipeline([
    ("preprocess", preprocessor_knn),
    ("clf", KNeighborsClassifier()),
])

pipe_dt = Pipeline([
    ("preprocess", preprocessor_dt),
    ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
])

## 14. GridSearchCV

Este bloco executa a busca de hiperparâmetros com validação cruzada estratificada. O critério de seleção é `average_precision`, pois o objetivo prático é ranquear e identificar melhor a classe minoritária (`TARGET=1`).

As grades foram mantidas conservadoras para que o notebook rode no Colab sem custo excessivo, especialmente no KNN.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

if FAST_MODE:
    param_grid_knn = {
        "clf__n_neighbors": [11, 21],
        "clf__metric": ["manhattan"],
        "clf__weights": ["distance"],
    }
    param_grid_dt = {
        "clf__max_depth": [5, 10],
        "clf__min_samples_split": [10, 50],
        "clf__criterion": ["gini"],
    }
else:
    param_grid_knn = {
        "clf__n_neighbors": [11, 21],
        "clf__metric": ["euclidean", "manhattan"],
        "clf__weights": ["uniform", "distance"],
    }
    param_grid_dt = {
        "clf__max_depth": [5, 10, 15, None],
        "clf__min_samples_split": [2, 10, 50],
        "clf__criterion": ["gini", "entropy"],
    }

gs_knn = GridSearchCV(pipe_knn, param_grid_knn, cv=cv, scoring="average_precision", n_jobs=-1, refit=True, verbose=1)
gs_dt = GridSearchCV(pipe_dt, param_grid_dt, cv=cv, scoring="average_precision", n_jobs=-1, refit=True, verbose=1)

dummy.fit(X_train, y_train)
gs_knn.fit(X_train, y_train)
gs_dt.fit(X_train, y_train)

print(f"Melhor KNN | AP médio CV: {gs_knn.best_score_:.4f} | params: {gs_knn.best_params_}")
print(f"Melhor DT  | AP médio CV: {gs_dt.best_score_:.4f} | params: {gs_dt.best_params_}")

## 15. Resultados da Validação Cruzada

Este bloco organiza os resultados do grid. A tabela permite discutir o efeito dos hiperparâmetros e não apenas apresentar o melhor modelo final.

In [ ]:
def grid_results_table(grid, model_name: str) -> pd.DataFrame:
    result = pd.DataFrame(grid.cv_results_).copy()
    param_cols = [col for col in result.columns if col.startswith("param_")]
    cols = ["rank_test_score", "mean_test_score", "std_test_score"] + param_cols
    result = result[cols].sort_values("rank_test_score").reset_index(drop=True)
    result.insert(0, "modelo", model_name)
    return result

knn_results = grid_results_table(gs_knn, "KNN")
dt_results = grid_results_table(gs_dt, "DecisionTree")

display(Markdown("### Top KNN")); display(knn_results.head(10))
display(Markdown("### Top Árvore de Decisão")); display(dt_results.head(10))

## 16. Avaliação Final no Holdout

Este bloco avalia os modelos no conjunto de teste, usado pela primeira vez aqui. Reportamos métricas completas, com destaque para `recall`, `precision` e `F1` da classe inadimplente.

In [ ]:
def get_positive_proba(model, X_eval):
    proba = model.predict_proba(X_eval)
    if proba.shape[1] == 1:
        only_class = model.classes_[0]
        return np.ones(len(X_eval)) if only_class == 1 else np.zeros(len(X_eval))
    positive_index = list(model.classes_).index(1)
    return proba[:, positive_index]


def evaluate_model(name: str, model, X_eval, y_eval, threshold: float = 0.5) -> dict:
    y_proba = get_positive_proba(model, X_eval)
    y_pred = (y_proba >= threshold).astype(int)
    metrics = {
        "modelo": name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision_1": precision_score(y_eval, y_pred, zero_division=0),
        "recall_1": recall_score(y_eval, y_pred, zero_division=0),
        "f1_1": f1_score(y_eval, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, y_proba),
        "average_precision": average_precision_score(y_eval, y_proba),
    }
    display(Markdown(f"### {name}")); display(pd.DataFrame([metrics]))
    print(classification_report(y_eval, y_pred, target_names=["Adimplente (0)", "Inadimplente (1)"], zero_division=0))
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(y_eval, y_pred, display_labels=["Adimplente", "Inadimplente"], ax=ax, cmap="Blues")
    ax.set_title(f"Matriz de confusão — {name}"); plt.show()
    return metrics

final_metrics = []
final_metrics.append(evaluate_model("DummyClassifier", dummy, X_test, y_test))
final_metrics.append(evaluate_model("KNN — melhor GridSearchCV", gs_knn, X_test, y_test))
final_metrics.append(evaluate_model("Árvore de Decisão — melhor GridSearchCV", gs_dt, X_test, y_test))

final_metrics_df = pd.DataFrame(final_metrics).sort_values("average_precision", ascending=False)
display(Markdown("## Ranking Final no Holdout")); display(final_metrics_df)

## 17. Curvas ROC e Precision-Recall

Este bloco mostra as curvas finais. A ROC-AUC mede separação geral; a curva Precision-Recall é mais importante para este problema porque `TARGET=1` é minoria.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, model in [("Dummy", dummy), ("KNN", gs_knn), ("DecisionTree", gs_dt)]:
    y_proba = get_positive_proba(model, X_test)
    RocCurveDisplay.from_predictions(y_test, y_proba, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, y_proba, name=name, ax=axes[1])
axes[0].set_title("Curva ROC — holdout")
axes[1].set_title("Curva Precision-Recall — holdout")
plt.tight_layout(); plt.show()

## 18. Sensibilidade ao Threshold

Este bloco mostra como precision, recall e F1 mudam quando alteramos o limiar de classificação.

A análise é feita para KNN e Árvore de Decisão, porque os resultados iniciais mostraram comportamentos muito diferentes: o KNN tende a prever quase tudo como adimplente, enquanto a árvore aumenta bastante o recall de inadimplentes ao custo de muitos falsos positivos.

Esta análise é diagnóstica. O threshold final de negócio deveria ser escolhido em validação ou por custo de erro, não olhando repetidamente o teste.

In [ ]:
def threshold_report(model_name, model, X_eval, y_eval):
    y_proba = get_positive_proba(model, X_eval)
    rows = []
    for threshold in np.arange(0.10, 0.91, 0.05):
        y_pred_thr = (y_proba >= threshold).astype(int)
        rows.append({
            "modelo": model_name,
            "threshold": round(float(threshold), 2),
            "precision_1": precision_score(y_eval, y_pred_thr, zero_division=0),
            "recall_1": recall_score(y_eval, y_pred_thr, zero_division=0),
            "f1_1": f1_score(y_eval, y_pred_thr, zero_division=0),
            "predicted_positive_rate": float(y_pred_thr.mean()),
        })
    return pd.DataFrame(rows)

threshold_df = pd.concat([
    threshold_report("KNN", gs_knn, X_test, y_test),
    threshold_report("Árvore de Decisão", gs_dt, X_test, y_test),
], ignore_index=True)

plt.figure(figsize=(11, 5))
sns.lineplot(data=threshold_df, x="threshold", y="precision_1", hue="modelo", marker="o")
plt.title("Precision TARGET=1 por threshold")
plt.ylim(0, 1)
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(data=threshold_df, x="threshold", y="recall_1", hue="modelo", marker="o")
plt.title("Recall TARGET=1 por threshold")
plt.ylim(0, 1)
plt.show()

plt.figure(figsize=(11, 5))
sns.lineplot(data=threshold_df, x="threshold", y="f1_1", hue="modelo", marker="o")
plt.title("F1 TARGET=1 por threshold")
plt.ylim(0, 1)
plt.show()

display(threshold_df.sort_values(["modelo", "f1_1"], ascending=[True, False]).groupby("modelo").head(5))

## 19. Conclusão para o Relatório

Ao final da execução, discuta:

1. Por que `accuracy` não é métrica decisiva em uma base 90/10.
2. Por que `average_precision` foi usada como critério de seleção.
3. Por que `train.csv` bruto evita leakage e double-scaling.
4. Por que sentinelas podem ser convertidos antes do split, mas imputação e escala devem ficar dentro do pipeline.
5. Por que variáveis sensíveis foram removidas da modelagem.
6. O resultado da validação sobre flag `CASA`, scaler e `ANOSULTIMADECLARACAO`.
7. Qual modelo venceu no holdout e quais trade-offs apresentou para a classe inadimplente.

Evite conclusões causais. O modelo identifica padrões estatísticos associados à inadimplência, não causas sociais ou individuais.